In [0]:
%pip install gradio prophet openai

INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 86.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Not uninstalling click at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6ea734d1-a75c-4bfc-a2f5-e483654b6de1
    Can't uninstall 'click'. No files were found to uninstall.
  Attempting uninstall: starlette
    Found existing installation: starlette 0.48.0
    Not uninstalling starlette at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6ea734d1-a75c-4bfc-a2f5-e483654b6de1
    Can't uninstall 'starlette'. 

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lag, when
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ConstructoraDataOps").getOrCreate()

# 1. Capa Bronze: Conexión a la tabla cargada manualmente en Unity Catalog
df_bronze = spark.table("proyecto_gestion_costos_operativos.default.historico_equipos")

# 2. Capa Silver: Limpieza y estandarización
df_silver = df_bronze \
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd")) \
    .dropna()

# Guardado automático de la tabla Silver en el catálogo
df_silver.write.format("delta").mode("overwrite").saveAsTable("proyecto_gestion_costos_operativos.default.silver_equipos")

# 3. Capa Gold: Ingeniería de características (Features)
windowSpec = Window.orderBy("Date")

# CORRECCIÓN: las columnas Alerta_Sobrecosto se calculan AHORA dentro de la
# misma cadena, antes del único .dropna() final. En la versión original el
# .dropna() se aplicaba antes de crear estas columnas, y como también usan
# lag(), la primera fila del dataset quedaba con valores nulos en el target
# que Notebook 2 usa para entrenar (RandomForestClassifier falla o distorsiona
# métricas con un target nulo).
df_gold = spark.table("proyecto_gestion_costos_operativos.default.silver_equipos") \
    .withColumn("Price_X_lag30", lag("Price_X", 30).over(windowSpec)) \
    .withColumn("Price_Y_lag30", lag("Price_Y", 30).over(windowSpec)) \
    .withColumn("Price_Z_lag30", lag("Price_Z", 30).over(windowSpec)) \
    .withColumn(
        "Alerta_Sobrecosto_Eq1",
        when(col("Price_Equipo1") > lag("Price_Equipo1", 1).over(windowSpec), 1).otherwise(0)
    ) \
    .withColumn(
        "Alerta_Sobrecosto_Eq2",
        when(col("Price_Equipo2") > lag("Price_Equipo2", 1).over(windowSpec), 1).otherwise(0)
    ) \
    .dropna()

# Guardado automático de la tabla Gold en el catálogo
df_gold.write.format("delta").mode("overwrite").saveAsTable("proyecto_gestion_costos_operativos.default.gold_equipos_features")